In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("KafkaToDB") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate() 

spark.sparkContext.setLogLevel("WARN")


25/07/07 15:01:31 WARN Utils: Your hostname, nader-IdeaPad resolves to a loopback address: 127.0.1.1; using 10.72.0.97 instead (on interface wlp0s20f3)
25/07/07 15:01:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/nader/miniconda3/envs/bigdata_env/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/nader/.ivy2/cache
The jars for the packages stored in: /home/nader/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8cab6734-79b4-4cb4-b1dc-d056c01d92d5;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 408ms :: artifacts dl 13ms
	:: m

In [8]:
df_raw = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "github-repos") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()




In [9]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

metadata_schema = StructType([
    StructField("name", StringType()),
    StructField("owner", StringType()),
    StructField("created_at", StringType()),
    StructField("updated_at", StringType()),
    StructField("language", StringType()),
    StructField("stars", IntegerType())
])

root_schema = StructType([
    StructField("metadata", metadata_schema),
    StructField("collected_at", StringType())
])

df_value = df_raw.selectExpr("CAST(value AS STRING) as raw_value")

df_parsed = df_value.select(from_json(col("raw_value"), root_schema).alias("data"))

df_flat = df_parsed.select(
    col("data.metadata.name").alias("repo_name"),
    col("data.metadata.owner").alias("repo_owner"),
    col("data.metadata.language").alias("language"),
    col("data.metadata.stars").alias("stars"),
    col("data.collected_at").alias("collected_at")
)

# Now show the results once
df_flat.show(truncate=False)


25/07/07 11:02:45 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


+--------------------+----------+--------+-----+--------------------------+
|repo_name           |repo_owner|language|stars|collected_at              |
+--------------------+----------+--------+-----+--------------------------+
|Base-de-connaissance|Git-Know  |NULL    |0    |2025-07-07T10:52:54.090764|
|Base-de-connaissance|Git-Know  |NULL    |0    |2025-07-07T10:53:23.562294|
|metasfresh          |Git-Know  |NULL    |0    |2025-07-07T10:54:30.598288|
|evershop            |Git-Know  |NULL    |0    |2025-07-07T10:55:33.405495|
|maybe               |Git-Know  |NULL    |0    |2025-07-07T10:56:36.178387|
|shop-e-commerce     |Git-Know  |NULL    |0    |2025-07-07T10:57:34.354184|
+--------------------+----------+--------+-----+--------------------------+



In [7]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import *

df_raw_commits = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "github-commits") \
    .option("startingOffsets", "earliest") \
    .load()


from pyspark.sql.functions import from_json, col, explode
from pyspark.sql.types import *

# === Define Schemas ===

file_schema = StructType([
    StructField("filename", StringType()),
    StructField("status", StringType()),
    StructField("additions", IntegerType()),
    StructField("deletions", IntegerType())
])

commit_schema = StructType([
    StructField("sha", StringType()),
    StructField("commit", StructType([
        StructField("author", StructType([
            StructField("name", StringType()),
            StructField("email", StringType()),
            StructField("date", StringType()),
        ])),
        StructField("message", StringType()),
    ])),
    StructField("stats", StructType([
        StructField("total", IntegerType()),
        StructField("additions", IntegerType()),
        StructField("deletions", IntegerType()),
    ])),
    StructField("files", ArrayType(file_schema)),

])

# === Parse JSON from Kafka or Raw Source ===

df_value = df_raw_commits.selectExpr("CAST(value AS STRING) as raw_value")

df_parsed = df_value.select(from_json(col("raw_value"), commit_schema).alias("data"))

# === Flatten the data, including commit and file changes ===

df_flat = df_parsed \
    .withColumn("file", explode(col("data.files"))) \
    .select(
        col("data.sha").alias("commit_sha"),
        col("data.commit.author.name").alias("author_name"),
        col("data.commit.author.email").alias("author_email"),
        col("data.commit.author.date").alias("commit_date"),
        col("data.commit.message").alias("commit_message"),
        col("data.stats.total").alias("total_changes"),
        col("data.stats.additions").alias("total_additions"),
        col("data.stats.deletions").alias("total_deletions"),
        col("file.filename").alias("file_name"),
        col("file.status").alias("file_status"),
        col("file.additions").alias("file_additions"),
        col("file.deletions").alias("file_deletions"),

    )

# === Show the cleaned and structured commit data ===
df_flat.show(truncate=False)


+----------------------------------------+--------------+--------------------------------+--------------------+-----------------------------------------------------------------------------------+-------------+---------------+---------------+--------------------------------------------------+-----------+--------------+--------------+
|commit_sha                              |author_name   |author_email                    |commit_date         |commit_message                                                                     |total_changes|total_additions|total_deletions|file_name                                         |file_status|file_additions|file_deletions|
+----------------------------------------+--------------+--------------------------------+--------------------+-----------------------------------------------------------------------------------+-------------+---------------+---------------+--------------------------------------------------+-----------+--------------+-----------

25/07/07 15:13:14 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
